# VLM 智能图文问答助手 - Kaggle 评测
---
**用途**: 在 Kaggle Notebook 中一键运行完整评测流水线

**前置**: 已将 project.zip 上传为 Kaggle Dataset 或直接上传 Notebook

**预计耗时**: VQA-v2 1000条 + TextVQA 500条 ≈ 30-60分钟 (T4 GPU)

In [ ]:
# ============================================================
# Cell 1: 环境准备
# ============================================================
import os
import sys
import subprocess

# 安装依赖
!pip install -q transformers accelerate bitsandbytes qwen-vl-utils datasets pillow pyyaml tqdm pandas easyocr

# 如果项目代码在 Kaggle Dataset 中
# 假设将 project.zip 上传为 dataset，挂载到 /kaggle/input/vlm-project/
import zipfile
project_path = "/kaggle/working/vlm_project"

# 尝试找到 project.zip
zip_candidates = [
    "/kaggle/input/vlm-project/project.zip",
    "/kaggle/working/project.zip",
]

found = False
for zp in zip_candidates:
    if os.path.exists(zp):
        print(f"[*] 找到 project.zip: {zp}")
        with zipfile.ZipFile(zp, 'r') as zf:
            os.makedirs(project_path, exist_ok=True)
            zf.extractall(project_path)
        found = True
        break

if not found:
    # 如果 notebook 和项目代码在同一个 Git 仓库中，直接 clone
    print("[*] 未找到 project.zip，尝试从 GitHub clone...")
    # 替换为你的实际仓库地址
    # !git clone https://github.com/yourname/vlm-vqa-assistant.git {project_path}
    print("[!] 请将 project.zip 上传为 Kaggle Dataset，或将代码 clone 到此。")

if os.path.exists(project_path):
    sys.path.insert(0, project_path)
    os.chdir(project_path)
    print(f"[OK] 项目路径: {project_path}")
    !ls -la {project_path}
else:
    print("[ERROR] 项目代码未就绪")

print(f"[INFO] GPU 信息:")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# ============================================================
# Cell 2: Baseline 评测（通用 Prompt）
# ============================================================

# 确保在项目根目录
os.chdir(project_path)

# 运行通用模板 baseline
!python experiments/scripts/run_eval.py \
    --template general \
    --datasets vqa_v2,textvqa

In [ ]:
# ============================================================
# Cell 3: TextVQA 专用 Prompt 评测
# ============================================================

!python experiments/scripts/run_eval.py \
    --template textvqa \
    --datasets textvqa

In [ ]:
# ============================================================
# Cell 4: 场景化 Prompt 评测
# ============================================================

!python experiments/scripts/run_eval.py \
    --template scene \
    --datasets vqa_v2

In [ ]:
# ============================================================
# Cell 5: 查看结果
# ============================================================

import json
import glob

result_files = sorted(glob.glob("experiments/results/*.json"))
print(f"找到 {len(result_files)} 个结果文件:\n")

for rf in result_files[-3:]:  # 最近 3 个
    with open(rf, 'r') as f:
        data = json.load(f)
    print(f"{'='*50}")
    print(f"文件: {os.path.basename(rf)}")
    print(json.dumps(data, ensure_ascii=False, indent=2))
    print()

In [ ]:
# ============================================================
# Cell 6: 下载结果（Kaggle 限制直接下载，打包后用右侧面板下载）
# ============================================================

import shutil
shutil.make_archive("/kaggle/working/results", 'zip', "experiments/results")
print("[OK] 结果已打包为 /kaggle/working/results.zip，请在右侧 Output 面板下载")